# Notebook 02 — Preprocessing

**Project:** Wafer Defect Detection & Yield Risk Dashboard  
**Phase:** Phase 2 — Data Preprocessing

## Goals

1. Load the labeled WM-811K samples using `src/data_loader.py`
2. Resize all wafer maps to 64×64 using nearest-neighbor interpolation
3. Normalize pixel values from {0,1,2} to {0.0, 0.5, 1.0}
4. Encode string labels to integer indices
5. Create a stratified 70/15/15 train/val/test split
6. Verify the split is balanced across classes
7. Save processed arrays to `data/processed/`
8. Compute class weights for the loss function

## Why This Step Matters

Neural networks require fixed-size, normalized inputs.
WM-811K has 129 unique wafer map heights — we must standardize to one shape.
The 989x class imbalance means we also need class weights ready for training.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

# Allow importing from src/ when running notebook from notebooks/
sys.path.insert(0, str(Path('..').resolve()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap

from src.data_loader import load_labeled, LABEL_MAP, LABEL_MAP_INV, DEFECT_CLASSES
from src.preprocessing import (
    build_arrays, split_arrays, save_processed,
    compute_class_weights, TARGET_SIZE,
)

DATASET_PATH = Path('../data/raw/LSWMD.pkl')
OUTPUT_DIR = Path('../outputs/figures')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Label map:', LABEL_MAP)
print('Target size:', TARGET_SIZE)

## 2. Load Labeled Data

In [ ]:
df = load_labeled(DATASET_PATH)
print(f'\nClass distribution:')
print(df['label'].value_counts())

## 3. Visualize Resize Effect

Before running the full pipeline, let's verify that resizing to 64×64 preserves the defect pattern structure.

In [ ]:
import cv2
from src.preprocessing import resize_wafer_map, normalize_wafer_map

wafer_cmap = ListedColormap(['black', '#4CAF50', '#F44336'])

# Pick one sample per class and show original vs resized
fig, axes = plt.subplots(2, 9, figsize=(18, 5))
fig.suptitle('Original (top) vs Resized 64×64 (bottom)', fontsize=13, fontweight='bold')

for col, cls in enumerate(DEFECT_CLASSES):
    subset = df[df['label'] == cls]
    if len(subset) == 0:
        axes[0, col].axis('off')
        axes[1, col].axis('off')
        continue

    original = np.array(subset.iloc[0]['waferMap'])
    resized = resize_wafer_map(original)

    axes[0, col].imshow(original, cmap=wafer_cmap, vmin=0, vmax=2, interpolation='nearest')
    axes[0, col].set_title(cls, fontsize=8)
    axes[0, col].axis('off')

    axes[1, col].imshow(resized, cmap=wafer_cmap, vmin=0, vmax=2, interpolation='nearest')
    axes[1, col].set_title('64×64', fontsize=8)
    axes[1, col].axis('off')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'resize_comparison.png', dpi=150)
plt.show()
print('Saved: outputs/figures/resize_comparison.png')

## 4. Build Feature Matrix

This processes all 172,950 labeled wafer maps.

- Input: 2D array (variable size), values {0, 1, 2}
- Output: 4D array shape (N, 1, 64, 64), values {0.0, 0.5, 1.0}

Expected time: ~2–4 minutes.

In [ ]:
print('Building feature arrays (this may take a few minutes)...')
X, y = build_arrays(df, size=TARGET_SIZE, verbose=True)

print(f'\nX shape: {X.shape}  dtype: {X.dtype}')
print(f'y shape: {y.shape}  dtype: {y.dtype}')
print(f'X value range: [{X.min():.1f}, {X.max():.1f}]')
print(f'Unique y values: {np.unique(y)}')
print(f'Memory usage: X = {X.nbytes / 1e9:.2f} GB')

## 5. Stratified Train / Val / Test Split

In [ ]:
X_train, X_val, X_test, y_train, y_val, y_test = split_arrays(X, y)

print('Split sizes:')
print(f'  Train: {len(X_train):,}  ({len(X_train)/len(X)*100:.1f}%)')
print(f'  Val:   {len(X_val):,}   ({len(X_val)/len(X)*100:.1f}%)')
print(f'  Test:  {len(X_test):,}   ({len(X_test)/len(X)*100:.1f}%)')

## 6. Verify Class Balance Across Splits

Stratification ensures rare classes (Near-full: 149 total) appear in all three splits.

In [ ]:
splits = {'train': y_train, 'val': y_val, 'test': y_test}
rows = []
for cls_idx, cls_name in LABEL_MAP_INV.items():
    row = {'Class': cls_name}
    for split_name, y_split in splits.items():
        cnt = (y_split == cls_idx).sum()
        row[f'{split_name}_n'] = cnt
        row[f'{split_name}_%'] = f"{cnt / len(y_split) * 100:.1f}"
    rows.append(row)

split_df = pd.DataFrame(rows)
print('Class distribution across splits:')
print(split_df.to_string(index=False))

In [ ]:
# Visualize split distribution
x_pos = np.arange(len(DEFECT_CLASSES))
width = 0.25

fig, ax = plt.subplots(figsize=(13, 5))
ax.bar(x_pos - width, [split_df[split_df['Class']==c]['train_n'].values[0] for c in DEFECT_CLASSES], width, label='Train', color='steelblue')
ax.bar(x_pos,          [split_df[split_df['Class']==c]['val_n'].values[0]   for c in DEFECT_CLASSES], width, label='Val',   color='orange')
ax.bar(x_pos + width,  [split_df[split_df['Class']==c]['test_n'].values[0]  for c in DEFECT_CLASSES], width, label='Test',  color='green')

ax.set_yscale('log')
ax.set_title('Class Distribution Across Splits (log scale)', fontsize=13, fontweight='bold')
ax.set_xlabel('Defect Class')
ax.set_ylabel('Sample Count (log)')
ax.set_xticks(x_pos)
ax.set_xticklabels(DEFECT_CLASSES, rotation=25)
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'split_distribution.png', dpi=150)
plt.show()
print('Saved: outputs/figures/split_distribution.png')

## 7. Class Weights for Loss Function

In [ ]:
weights = compute_class_weights(y_train)

print('Class weights for CrossEntropyLoss:')
for idx, cls in LABEL_MAP_INV.items():
    print(f'  [{idx}] {cls:<15}  weight = {weights[idx]:.4f}')

print(f'\nMin weight: {weights.min():.4f} ({DEFECT_CLASSES[weights.argmin()]})')
print(f'Max weight: {weights.max():.4f} ({DEFECT_CLASSES[weights.argmax()]})')

## 8. Save Processed Data

In [ ]:
save_processed(X_train, X_val, X_test, y_train, y_val, y_test, out_dir='../data/processed')

## 9. Sanity Check — Load and Verify

In [ ]:
from src.preprocessing import load_processed

X_tr, X_v, X_te, y_tr, y_v, y_te = load_processed('../data/processed')
print('Loaded from disk:')
print(f'  X_train: {X_tr.shape}  y_train: {y_tr.shape}')
print(f'  X_val:   {X_v.shape}   y_val:   {y_v.shape}')
print(f'  X_test:  {X_te.shape}  y_test:  {y_te.shape}')

# Visualize a few samples from the training set
wafer_cmap = ListedColormap(['black', '#4CAF50', '#F44336'])
fig, axes = plt.subplots(1, 9, figsize=(18, 3))
fig.suptitle('One Training Sample per Class (after preprocessing)', fontsize=11, fontweight='bold')

for cls_idx, cls_name in LABEL_MAP_INV.items():
    mask = y_tr == cls_idx
    if not mask.any():
        axes[cls_idx].axis('off')
        continue
    sample = X_tr[mask][0, 0]  # shape (64, 64)
    # Convert back to {0,1,2} for display
    display = (sample * 2).round().astype(int)
    axes[cls_idx].imshow(display, cmap=wafer_cmap, vmin=0, vmax=2, interpolation='nearest')
    axes[cls_idx].set_title(cls_name, fontsize=8)
    axes[cls_idx].axis('off')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'preprocessed_samples.png', dpi=150)
plt.show()
print('Saved: outputs/figures/preprocessed_samples.png')

## 10. Summary

| Property | Value |
|---|---|
| Input wafer maps | 172,950 labeled samples |
| Resize target | 64×64 (nearest-neighbor) |
| Normalization | {0,1,2} → {0.0, 0.5, 1.0} |
| Array shape | (N, 1, 64, 64) — channel-first |
| Train split | ~121,065 (70%) |
| Val split | ~25,943 (15%) |
| Test split | ~25,942 (15%) |
| Split strategy | Stratified by class |
| Class weights | Computed, ready for CrossEntropyLoss |
| Saved to | `data/processed/` |

**Next step:** `notebooks/03_baseline_model.ipynb`